
# Ex | Resilience {.unnumbered}

Open the latest version on the [*web*](https://wbarfuss.github.io/csm-of-hei/02.03ex-Resilience.html), [*Github*](https://github.com/wbarfuss/csm-of-hei/blob/main/02.03ex-Resilience.ipynb) or in [*GoogleColab*](https://colab.research.google.com/github/wbarfuss/csm-of-hei/blob/main/02.03ex-Resilience.ipynb) 

> In these exercises, we apply the resilience concepts of the lecture to a second tipping element model, test how much we can trust the early-warning indicator, and investigate how heavy-tailed shocks change resilience and its early warning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

import matplotlib.style as style; style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (7.8, 2.5)
plt.rcParams['axes.facecolor'] = 'white'; plt.rcParams['grid.color'] = 'gray'; plt.rcParams['grid.linewidth'] = 0.25; 

From the lecture, we reuse the tipping element model with shocks, its equilibrium points, the critical value of $h$, and the simulation of a ramp of $h$ towards the tipping point.

In [ ]:
def F_tipmod(x, h, r, tau): return x + (h + r*x - x**3)/tau
def F_tipmod_noise(x, h, r, tau, n, rng):
    return F_tipmod(x, h, r, tau) + n*rng.standard_normal(np.shape(x))

def equilibria(h, r=1):  # real roots of h + r*x - x**3 = 0, sorted
    roots = np.roots([-1, 0, r, h])  # polynomial coefficients
    return np.sort(roots[np.abs(roots.imag) < 1e-9].real) # keep only real part

def h_crit(r=1): return 2*r/3*np.sqrt(r/3)  # critical value of h (see Tipping Elements)

def simulate_ramp(seed, xinit=1.1, iters=1000, n=0.125):
    rng = np.random.default_rng(seed)  # random number generator; the seed makes the result reproducible
    h = 0  # initial value of the external influence
    trajectory = []  # container to store the trajectory
    x = xinit  # re-storing the initial values
    for t in range(iters):  # looping through the iterations
        x_ = F_tipmod_noise(x, h, r=1, tau=2, n=n, rng=rng)
        trajectory.append(x_)  # storing the new state in the container
        x = x_  # the new state of the system `x_` becomes the current state `x`
        h -= 0.00038  # we slightly decrease h (i.e., reduce the resilience of the upper equilibrium point)
    return np.array(trajectory)

## Resilience in another tipping element

In the exercises on [Tipping Elements](https://wbarfuss.github.io/csm-of-hei/02.02ex-TippingElements.html), you analyzed a tipping element with different functional forms of the feedback loops,
$$\Delta x = \frac{1}{\tau}\left(h + rx^3 - x^5\right).$$
We now apply the resilience concepts of the lecture to this model. Use $r=1$ and $\tau=3$ throughout[^tau].

[^tau]: We use $\tau=3$ instead of $\tau=2$ as in the lecture. The wall of the upper cup rises steeply because of the $x^5$ term. With $\tau=2$, one time step throws the state from high up the wall past the bottom and over the edge, and upward pulses larger than about $0.35$ already tip the system (compare the footnote on overshoot in the lecture).

### Step 1 | Model

Implement the update function $F(x) = x + \Delta x$ with shocks of strength $n$, analogous to `F_tipmod_noise`, and a function that returns the sorted real equilibrium points, analogous to `equilibria`.

*Hint:* The equilibrium points are the real roots of the polynomial $-x^5 + rx^3 + h$, with coefficients `[-1, 0, r, 0, 0, h]`.

Compute the critical value $h_{crit}$ at which the upper cup disappears, and plot the potential function for $h = 0$, $h=-0.1$, and $h=-0.18$, together with the equilibrium points.

In [ ]:
# ...

### Step 2 | Ecological resilience

Compute the width of the upper cup for $h=-0.1$. Test your result with a single downward pulse slightly smaller and slightly larger than the width, analogous to `plot_pulse` in the lecture.

In [ ]:
# ...

### Step 3 | Basin stability

Compute the basin stability of the upper cup as a function of $h$ for the shock strengths $n=0.2$ and $n=0.3$, analogous to `basin_stability` in the lecture. Compare with the width from Step 2.

In [ ]:
# ...

### Step 4 | Engineering resilience

Derive $F'(x)$ for this model. Plot the width of the upper cup and the return rate $1-|F'(x_s)|$ as functions of $h$ in one plot, as in the lecture. Compare with the lecture model.

In [ ]:
# ...

### Step 5 | Early warning

Simulate a ramp of $h$ from $0$ to $-h_{crit}$ in $1000$ time steps, with shock strength $n=0.05$ and initial state $x=1$. Compute the lag-1 autocorrelation in sliding windows of $200$ time steps, as in the lecture. Does it rise before the system tips?

In [ ]:
# ...

### Step 6 | Conclusion

Which findings of the lecture hold for this model, and which numbers change?

...

## Robustness of the early-warning indicator

In the lecture, we computed the lag-1 autocorrelation for one realization of the shocks (seed 35). A real system also gives us only one realization. **How much can we trust a rise of AR(1)?** Use `simulate_ramp` from the lecture (above) throughout.

### Step 1 | Many realizations

Simulate the ramp for $200$ seeds. For each realization,
- find the time step at which the system tips, i.e., its state first drops below $0$;
- compute AR(1) in all windows of $200$ time steps that end at least $10$ time steps before the tipping;
- compute the rise of AR(1) from the first to the last of these windows.

Plot the AR(1) curves of all realizations in one plot, and a histogram of the time steps of tipping. What share of realizations shows a rise of more than $0.2$? Why do we stop the windows before the tipping?

*Hint:* `np.argmax(trajectory < 0)` returns the first index at which the condition holds.

In [ ]:
# ...

### Step 2 | False alarms

A rise of AR(1) is a warning only if it does not also occur when the resilience stays the same. Simulate $200$ realizations with $h$ fixed at $0$. For each seed, compute the rise of AR(1) over the same windows as for the ramp with the same seed in Step 1. What share of these realizations shows a rise of more than $0.2$?

In [ ]:
# ...

### Step 3 (optional) | Window length

Repeat Steps 1 and 2 for windows of $50$, $100$, $200$, and $400$ time steps. For each window length, compute the share of realizations with a rise above $0.2$ for the ramp (correct warnings) and for constant $h$ (false alarms), and the share of realizations that tip before the first window ends. Which window length would you choose?

In [ ]:
# ...

### Step 4 (optional) | Trend

A trend in the time series also raises the autocorrelation. In applications, we therefore remove the trend from each window before computing AR(1). Remove a linear trend from each window and compare AR(1) with and without detrending, for seed $35$ and the shock strengths $n=0.125$ and $n=0.03$. Use windows that end at time steps $200$ to $800$. When does detrending matter? Why?

*Hint:* `np.polyfit(t, x, 1)` fits a straight line, `np.polyval` evaluates it.

In [ ]:
# ...

### Step 5 | Conclusion

Summarize what your results mean for using the rise of AR(1) as an early-warning signal in a real system, of which we observe only one time series.

...

## Heavy-tailed shocks

So far, we assumed that the shocks are normally distributed with mean zero. Real-world shocks often have so-called **heavy tails**: large shocks are more probable than under a normal distribution.

The Student's t-distribution (or simply the t-distribution) is symmetric around zero and bell-shaped, like the normal distribution. It has one more parameter, the *degrees of freedom*, `df`. The smaller `df`, the heavier the tails; for `df` $\rightarrow \infty$, the t-distribution becomes the normal distribution.

For `df` $> 2$, the t-distribution has the variance $\mathrm{df}/(\mathrm{df}-2)$. We divide the samples by its square root, so that all shocks have unit variance, as in the lecture. Then only the shape of the distribution changes, not the size of the fluctuations. On a logarithmic axis, the difference in the tails is visible:

In [ ]:
from scipy.stats import t, norm
x = np.linspace(-6, 6, 301)
for df in [10, 4, 3]:
    scale = np.sqrt((df - 2)/df)  # scales the t-distribution to unit variance
    plt.plot(x, t.pdf(x, df, scale=scale), label=f'$t$, df $= {df}$')
plt.plot(x, norm.pdf(x), 'k--', label='Normal')
plt.yscale('log'); plt.ylim(1e-6, 1); plt.xlabel('Shock $\eta$'); plt.ylabel('PDF'); plt.legend();

### Step 1 | Shocks

Implement the update function of the lecture model with t-distributed shocks of unit variance, `F_tipmod_t(x, h, r, tau, n, df, rng)`. Plot one trajectory for normal shocks and one for $\mathrm{df}=3$, with $h=-0.2$, $r=1$, $\tau=2$, and $n=0.1$. Describe the difference.

*Hint:* `rng.standard_t(df, size)` draws samples from the t-distribution.

In [ ]:
# ...

### Step 2 | Basin stability

Compute the basin stability of the upper cup as a function of $h$ for normal shocks and for t-shocks with $\mathrm{df} \in \{10, 4, 3\}$, for the shock strengths $n=0.3$ and $n=0.5$. Do heavier tails make the system more or less resilient? Explain with the width of the cup.

In [ ]:
# ...

...

### Step 3 | Early warning

Repeat Step 1 of the exercise on the robustness of the early-warning indicator with t-shocks of unit variance, for $\mathrm{df} \in \{10, 4, 3\}$, and compare with normal shocks. For each distribution, compute the median time step of tipping, the share of realizations with a rise of AR(1) above $0.2$, and the share of realizations that tip before the first window ends.

Realizations that tip early have seen less of the ramp of $h$, so their autocorrelation has less time to rise, also with normal shocks. To separate this effect from the effect of the shocks, compute the share of rises above $0.2$ also for the realizations that tip after time step $600$ only.

In [ ]:
# ...

### Step 4 | Interpretation

Why does the early warning fail more often with heavy-tailed shocks? Is it the pulse or the press that tips the system?

*Hint:* For each realization, compare the value of $h$ at which it tips with $-h_{crit}$, and the size of the last shock before the tipping with the width of the cup at that moment.

In [ ]:
# ...

...

### Step 5 | Conclusion

Summarize how heavy-tailed shocks change the resilience of the system and the reliability of the early-warning indicator.

...